# Unit 7 — SoccerTwos / MA-POCA (Google Colab)

**Important:** Hugging Face says the AI-vs-AI feature is non-functional. Training and model upload are still possible, but this task may not be accepted by the certificate service. The official lab estimates **4–8 hours**, which may exceed Colab free-session limits.

**Run all cells from top to bottom.** Training and Hub uploads require network access and your Hugging Face write token.


## Install Unity ML-Agents (Python 3.10.12)


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg libgl1 libglib2.0-0 libgomp1 libxcursor1 libxi6 libxrandr2 libxinerama1 >/dev/null
%pip -q install "huggingface_hub<1"
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
!./bin/micromamba create -y -p /content/hf-rl-unity -c conda-forge python=3.10.12 pip
!/content/hf-rl-unity/bin/python -m pip install -q "torch==2.2.2" "numpy<2" "mlagents==1.1.0" "huggingface_hub==0.25.2"
!/content/hf-rl-unity/bin/python -c "import sys, mlagents; print(sys.version, mlagents.__version__)"

## Download the official Linux SoccerTwos executable

The original course hosts the executable on Google Drive. If the download link becomes restricted, this is an external blocker and the game binary must be obtained from the [official lab](https://huggingface.co/learn/deep-rl-course/unit7/hands-on).


In [ ]:
%pip -q install gdown
!gdown --id 1KuqBKYiXiIcU4kNMqEzhgypuFP5_45CL -O SoccerTwos.zip
!mkdir -p soccer
!unzip -q -o SoccerTwos.zip -d soccer
!find soccer -type f \( -name 'SoccerTwos' -o -name 'SoccerTwos.x86_64' \) -exec chmod +x {} \;
!find soccer -type f \( -name 'SoccerTwos' -o -name 'SoccerTwos.x86_64' \)

## MA-POCA config


In [ ]:
from pathlib import Path
Path("config").mkdir(exist_ok=True)
Path("config/SoccerTwos.yaml").write_text("behaviors:\n  SoccerTwos:\n    trainer_type: poca\n    hyperparameters:\n      batch_size: 2048\n      buffer_size: 20480\n      learning_rate: 0.0003\n      beta: 0.005\n      epsilon: 0.2\n      lambd: 0.95\n      num_epoch: 3\n      learning_rate_schedule: constant\n    network_settings:\n      normalize: false\n      hidden_units: 512\n      num_layers: 2\n      vis_encode_type: simple\n    reward_signals:\n      extrinsic:\n        gamma: 0.99\n        strength: 1.0\n    keep_checkpoints: 5\n    max_steps: 5000000\n    time_horizon: 1000\n    summary_freq: 10000\n    self_play:\n      save_steps: 50000\n      team_change: 200000\n      swap_steps: 2000\n      window: 10\n      play_against_latest_model_ratio: 0.5\n      initial_elo: 1200.0\n")
print("Config saved")

## Train the team (may take 4–8 hours)


In [ ]:
!SOCCER_EXE="$(find soccer -type f \( -name SoccerTwos -o -name SoccerTwos.x86_64 \) | head -1)"; test -x "$SOCCER_EXE" && /content/hf-rl-unity/bin/mlagents-learn config/SoccerTwos.yaml --env="$SOCCER_EXE" --run-id=SoccerTwos --no-graphics

## Upload the trained model


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Use a Hugging Face token with Write permission.

In [ ]:
!/content/hf-rl-unity/bin/mlagents-push-to-hf --run-id=SoccerTwos --local-dir=results/SoccerTwos --repo-id=panuza007/poca-SoccerTwos --commit-message="Train SoccerTwos"

Verify that the Hub model includes `SoccerTwos.onnx` and the tag `ML-Agents-SoccerTwos`. This tag is required by the old AI-vs-AI system (which is currently disabled).
